# NS_QUALITY_STATS Calibration

Computes the positional standard deviations of the pre-event-bonus sigmoid output
for ST, CM, RW, and LW. These are the `NS_QUALITY_STATS` constants used in the
quality-scaled goal bonus.

**Mean is fixed at 6.0** — not computed from data. With real-world anchored means,
an average real-world player has z=0 everywhere, raw_score=0, and
sigmoid(0) = 6.0 by the asymmetric sigmoid formula. Deriving the mean from Valencia
would bake in a top-team bias that penalises mid-table saves.

**Std is computed from Valencia data** — it reflects within-game-engine spread of
player quality, which is independent of team strength.

In [1]:
import json
import math
import numpy as np
import pandas as pd
from pathlib import Path

project_root = Path("..").resolve().parent

# Config
matches_path   = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
weights_path   = project_root / "config" / "performance_weights.json"
means_stds_path = project_root / "config" / "performance_means_stds.json"

raw_matches = json.load(open(matches_path))
weights     = json.load(open(weights_path))
means_stds  = json.load(open(means_stds_path))

print(f"Loaded {len(raw_matches)} matches")

Loaded 155 matches


In [2]:
# ── Constants (must match match_ratings_service.py exactly) ──────────────────

H_BASE = 10.0
MIN_MINUTES = 10.0

DUMMY_WEIGHTS = {
    "passes": 15.0, "distance_covered": 15.0, "distance_sprinted": 15.0,
    "possession_won": 15.0, "possession_lost": 15.0,
    "dribbles": 30.0, "tackles": 30.0, "fouls_committed": 30.0, "offsides": 30.0,
    "goals": 45.0, "assists": 45.0, "shots": 45.0,
}
DEFAULT_DUMMY = 30.0

XT_POSITION_SCALARS = {
    "ST": 0.35, "RW": 0.35, "LW": 0.35, "CAM": 0.35, "CF": 0.35,
    "CM": 0.25, "RM": 0.25, "LM": 0.25, "RWB": 0.25, "LWB": 0.25,
    "RB": 0.25, "LB": 0.25, "CDM": 0.10, "CB": 0.10,
}

LOG_TRANSFORMED_STATS = {
    "goals_p90", "assists_p90", "non_goal_shots_p90",
    "offsides_p90", "fouls_committed_p90", "possession_won_p90", "possession_lost_p90",
}
NEGATIVE_STATS = {"fouls_committed_p90", "possession_lost_p90", "offsides_p90"}

LOG_PRIOR_STATS = {"possession_won", "possession_lost", "fouls_committed", "offsides"}

VOLUME_MASKS = {
    "pass_accuracy_z":        ("passes",   3.0, 1.1),
    "dribble_success_rate_z": ("dribbles", 2.0, 1.5),
    "shot_accuracy_z":        ("shots",    1.5, 2.0),
    "tackle_success_rate_z":  ("tackles",  1.5, 2.0),
}

COL_NAMES = [
    "goals_p90", "assists_p90", "non_goal_shots_p90", "shot_accuracy",
    "passes_p90", "pass_accuracy", "dribbles_p90", "dribble_success_rate",
    "tackles_p90", "tackle_success_rate", "offsides_p90", "fouls_committed_p90",
    "possession_won_p90", "possession_lost_p90", "distance_covered_p90",
    "distance_sprinted_p90", "xt_bonus_p90",
]

# Positions that use the quality-scaled goal bonus
TARGET_POSITIONS = {"ST", "CM", "RW", "LW"}

# Fixed mean — derived from first principles:
# anchored means → average player z=0 everywhere → raw_score=0 → sigmoid(0) = 6.0
NS_MEAN_FIXED = 6.0

In [3]:
# ── Pipeline functions (mirrors match_ratings_service.py) ────────────────────

def apply_sigmoid(raw_score: float) -> float:
    """Asymmetric sigmoid: maps 0 → 6.0. k=0.85 above zero, k=0.45 below."""
    k = 0.85 if raw_score >= 0 else 0.45
    s_0 = math.log(2 / 3) / k
    return 10.0 * (1.0 / (1.0 + math.exp(-k * (raw_score - s_0))))


def apply_bayesian_smoothing(
    norm_metrics: dict, pos: str, minutes_played: float
) -> dict:
    """Replicate _apply_bayesian_smoothing from the service."""
    p90: dict = {}

    for col in ["goals", "assists", "shots"]:
        d = DUMMY_WEIGHTS.get(col, DEFAULT_DUMMY)
        p90[f"{col}_p90"] = (norm_metrics.get(col, 0.0) / (minutes_played + d)) * 90.0

    for col in ["passes", "dribbles", "tackles", "possession_won", "possession_lost",
                "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]:
        d = DUMMY_WEIGHTS.get(col, DEFAULT_DUMMY)
        stored_mean = means_stds.get(pos, {}).get(f"{col}_p90", {}).get("mean", 0.0)
        league_avg  = math.expm1(stored_mean) if col in LOG_PRIOR_STATS else stored_mean
        dummy_stat  = league_avg * (d / 90.0)
        p90[f"{col}_p90"] = (
            (norm_metrics.get(col, 0.0) + dummy_stat) / (minutes_played + d)
        ) * 90.0

    for col in ["shot_accuracy", "pass_accuracy", "dribble_success_rate", "tackle_success_rate"]:
        p90[col] = norm_metrics.get(col, 0.0)

    return p90


def calculate_xt_bonus(
    pos: str, pass_accuracy: float, passes_p90: float,
    distance_sprinted_p90: float, distance_covered_p90: float,
    possession_won_p90: float, possession_lost_p90: float,
) -> float:
    if distance_covered_p90 == 0:
        return 0.0
    base_scalar   = XT_POSITION_SCALARS.get(pos, 0.25)
    control_ratio = min((possession_won_p90 + 1.0) / (possession_lost_p90 + 1.0), 2.5)
    return (
        base_scalar
        * control_ratio
        * (distance_sprinted_p90 / distance_covered_p90)
        * math.log(pass_accuracy * passes_p90 + 1.0)
    )


def calculate_z_scores(
    p90: dict, pos: str, norm_metrics: dict
) -> dict:
    """Replicate _calculate_z_scores from the service."""
    pos_ms  = means_stds.get(pos, {})
    z_scores = {}

    for col in COL_NAMES:
        value = p90.get(col, 0.0)
        col_stats = pos_ms.get(col, {})
        mean = col_stats.get("mean", 0.0)
        std  = col_stats.get("std",  1.0)

        if col in LOG_TRANSFORMED_STATS:
            value = math.log1p(value)

        if std == 0.0:
            raw_z = 0.0
        elif col in NEGATIVE_STATS:
            raw_z = (mean - value) / std
        else:
            raw_z = (value - mean) / std

        z_key = f"{col}_z"
        if z_key in VOLUME_MASKS:
            vol_col, threshold, lam = VOLUME_MASKS[z_key]
            x_vol = norm_metrics.get(vol_col, 0.0)
            if x_vol <= 1.0:
                z_scores[z_key] = 0.0
                continue
            w_mask = 1.0 / (1.0 + math.exp(-lam * (x_vol - threshold)))
            z_scores[z_key] = raw_z * w_mask
        else:
            z_scores[z_key] = raw_z

    return z_scores


def calculate_base_sigmoid(norm_metrics: dict, pos: str, minutes_played: float) -> float:
    """Run the full pipeline up to the sigmoid, excluding any event bonus.

    Returns the sigmoid output of (base_dot_product * impact_scalar).
    This is the value NS_QUALITY_STATS is calibrated against.
    """
    p90 = apply_bayesian_smoothing(norm_metrics, pos, minutes_played)

    p90["non_goal_shots_p90"] = max(0.0, p90.get("shots_p90", 0.0) - p90.get("goals_p90", 0.0))

    p90["xt_bonus_p90"] = calculate_xt_bonus(
        pos=pos,
        pass_accuracy=norm_metrics.get("pass_accuracy", 0.0),
        passes_p90=p90.get("passes_p90", 0.0),
        distance_sprinted_p90=p90.get("distance_sprinted_p90", 0.0),
        distance_covered_p90=p90.get("distance_covered_p90", 0.0),
        possession_won_p90=p90.get("possession_won_p90", 0.0),
        possession_lost_p90=p90.get("possession_lost_p90", 0.0),
    )

    z_scores = calculate_z_scores(p90, pos, norm_metrics)

    pos_w    = weights.get(pos, {})
    dot_prod = sum(pos_w.get(col, 0.0) * z_scores.get(f"{col}_z", 0.0) for col in COL_NAMES)

    impact_scalar = math.sqrt(min(minutes_played, 90.0) / 90.0)
    raw_score     = dot_prod * impact_scalar

    return apply_sigmoid(raw_score)

In [4]:
# ── Run pipeline on all Valencia appearances ─────────────────────────────────

scores_by_pos: dict[str, list[float]] = {pos: [] for pos in TARGET_POSITIONS}

for match in raw_matches:
    data = match["data"]
    half_length = data["half_length"]
    time_scalar = H_BASE / half_length

    home_name = data["home_team_name"]
    is_valencia_home = (home_name == "Valencia CF")

    home_stats = data["home_stats"]
    away_stats = data["away_stats"]
    team_possession = home_stats["possession"] if is_valencia_home else away_stats["possession"]

    for perf in match["player_performances"]:
        if perf.get("performance_type") == "GK":
            continue

        positions = perf.get("positions_played", [])
        # Only keep pure single-position appearances for primary positions
        if len(positions) != 1 or positions[0] not in TARGET_POSITIONS:
            continue

        pos = positions[0]

        minutes_played = float(perf.get("minutes_played", 0))
        if minutes_played < MIN_MINUTES:
            continue

        # Numeric coercion
        raw = {k: float(v) if isinstance(v, (int, float)) else 0.0
               for k, v in perf.items()
               if k not in ["performance_type", "positions_played", "player_id", "match_id"]}

        # Half-length normalisation (volume stats only)
        vol_cols = ["passes", "dribbles", "shots", "tackles", "possession_won",
                    "possession_lost", "fouls_committed", "offsides",
                    "distance_covered", "distance_sprinted"]
        norm: dict = {k: v for k, v in raw.items()}
        for col in vol_cols:
            if col in norm:
                norm[col] = norm[col] * time_scalar

        # Possession adjustment (mirrors calibration notebooks)
        atk_cols = ["passes", "dribbles", "shots", "possession_lost"]
        def_cols = ["tackles", "possession_won", "fouls_committed"]
        for col in atk_cols:
            if col in norm:
                norm[col] *= math.sqrt(50.0 / team_possession)
        for col in def_cols:
            if col in norm:
                norm[col] *= math.sqrt(50.0 / (100.0 - team_possession))

        score = calculate_base_sigmoid(norm, pos, minutes_played)
        scores_by_pos[pos].append(score)

for pos, scores in scores_by_pos.items():
    print(f"{pos}: {len(scores)} appearances")

LW: 227 appearances
ST: 234 appearances
CM: 424 appearances
RW: 217 appearances


In [5]:
# ── Compute and display results ──────────────────────────────────────────────

print(f"Fixed mean (all positions): {NS_MEAN_FIXED}")
print()
print(f"{'Position':<10} {'N':>6} {'Computed mean':>14} {'Std':>8}")
print("-" * 44)

ns_quality_stats = {}
for pos in ["ST", "CM", "RW", "LW"]:
    scores = scores_by_pos[pos]
    if not scores:
        print(f"{pos:<10} {'0':>6} {'N/A':>14} {'N/A':>8}")
        continue
    computed_mean = float(np.mean(scores))
    std           = float(np.std(scores, ddof=1))
    ns_quality_stats[pos] = (NS_MEAN_FIXED, std)
    delta = computed_mean - NS_MEAN_FIXED
    print(f"{pos:<10} {len(scores):>6} {computed_mean:>14.4f} {std:>8.4f}   (Δ from fixed mean: {delta:+.4f})")

print()
print("NS_QUALITY_STATS output (mean fixed at 6.0):")
print("NS_QUALITY_STATS = {")
for pos, (mean, std) in ns_quality_stats.items():
    print(f'    "{pos}": ({mean}, {std:.6f}),')
print("}")

Fixed mean (all positions): 6.0

Position        N  Computed mean      Std
--------------------------------------------
ST            234         6.2238   0.6219   (Δ from fixed mean: +0.2238)
CM            424         6.1677   0.5585   (Δ from fixed mean: +0.1677)
RW            217         6.0480   0.4904   (Δ from fixed mean: +0.0480)
LW            227         6.1535   0.5176   (Δ from fixed mean: +0.1535)

NS_QUALITY_STATS output (mean fixed at 6.0):
NS_QUALITY_STATS = {
    "ST": (6.0, 0.621923),
    "CM": (6.0, 0.558504),
    "RW": (6.0, 0.490393),
    "LW": (6.0, 0.517578),
}
